In [1]:
# Load libraries
import pyhocon
from tqdm import tqdm
from itertools import combinations
from sklearn.utils import shuffle
from torch.utils import data
from torch.utils.data import DataLoader

from data_loader import MultiTaskBatchSampler
from models import FullCrossEncoder
from evaluator import Evaluation
from dataset import CrossEncoderDatasetFull,CrossEncoderMixedDatasetFull, ParaphraseDataset
from utils import *


C:\Users\sago\AppData\Roaming\Python\Python38\site-packages\transformers\generation_utils.py:24: FutureWarning: Importing `GenerationMixin` from `src/transformers/generation_utils.py` is deprecated and will be removed in Transformers v5. Import as `from transformers import GenerationMixin` instead.
  warnings.warn(
Xformers is not installed correctly. If you want to use memory_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


In [2]:
#Load config files
config = pyhocon.ConfigFactory.parse_file('configs/config_pairwise.json')
fix_seed(config)
logger = create_logger(config, create_file=True)
logger.info(pyhocon.HOCONConverter.convert(config, "hocon"))
create_folder(config.model_path)
print(config.experiment_id)

Experiment5


In [3]:
config_experiments = pyhocon.ConfigFactory.parse_file('configs/config_experiments.json')
datasets_config = pyhocon.ConfigFactory.parse_file('configs/config_datasets.json')
datasets_config = resolve_data_path(datasets_config)

### Load datasets according to config file

In [4]:
config.data_folder

'../../CDCR/data/ECBplus-prep/output_data/'

In [5]:
# init train and dev set
if "Experiment" in config.experiment_id:
    print("loads experiment")
    train = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['train'])
    val = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['val'])
    #test = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['test'])

else:
    if config.data_folder == "mrpc" or "HoW_DatasetDict" in config.data_folder:
        train = ParaphraseDataset(config, 'train') 
        val = ParaphraseDataset(config, 'validation')
    else:
        train = CrossEncoderDatasetFull(config, 'train') 
        val = CrossEncoderDatasetFull(config, 'val')

loads experiment
-----------------
dataset_key ECBPLUS_ENTITY_CONFIG
split train
negative_sampling True
mention_type entity_mentions


c:\Users\sago\Documents\GitHub\CDCR_lexical_diversity\MTL\cross_encoder_adapted\dataset.py:139: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels,dtype=torch.float)


num_mentions 6055
num_instances 209254
-----------------
dataset_key ECBPLUS_EVENT_CONFIG
split train
negative_sampling True
mention_type event_mentions
num_mentions 2892
num_instances 81189
-----------------
dataset_key FCC_T_ENTITY_CONFIG
split train
negative_sampling True
mention_type entity_mentions_attr
num_mentions 3975
num_instances 6625
-----------------
dataset_key FCC_T_EVENT_CONFIG
split train
negative_sampling True
mention_type event_mentions
topic 7CqELkejZUeV3JiDCpWSs6  has only one mention. Mentions pair not created for dataset  FCC-prep/event_mentions/train
topic QSTmTYGsbkA7nrrDBDVsA9  has only one mention. Mentions pair not created for dataset  FCC-prep/event_mentions/train
num_mentions 1602
num_instances 16073
-----------------
dataset_key MRPC
split train
negative_sampling False
num_instances 3668
-----------------
dataset_key HOW
split test
negative_sampling False
num_instances 30337
-----------------
dataset_key ECBPLUS_ENTITY_CONFIG
split val
negative_sampling Fa

In [6]:
#Debug sizes, splits


In [6]:
sampler = MultiTaskBatchSampler(        dataset=train,
        batch_size=32,
        mix_opt=1,
        extra_task_ratio=1,
        current_epoch=1,
        total_epochs=10,
        bin_size=64,
        bin_on=False,
        bin_grow_ratio=0.5,
        sampling = "sequential")
        
train_dataloader = DataLoader(
        train,
        batch_sampler=sampler,
        pin_memory=True
        )

In [8]:
from collections import Counter

for instances,labels,instances_id,task_ids in train_dataloader:
    print([extract_dataset_info_from_instance_id(instances_id) for instances_id in instances_id])
    break
    
    
    # dataset_names = []
    # for id in instances_id:
    #     parts = id.split("[SEP]", 1)
    #     sub_parts = parts[0].split("/")
    #     dataset_names.append(sub_parts[0])
    #     if 'how'==sub_parts[0]:
    #         print(sub_parts)
    #         break
    
    # value_counts = Counter(dataset_names)
    # output_strings = [f"{value}: {count}" for value, count in value_counts.items()]
    # print(', '.join(output_strings))



[{'mention_type': 'entity_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'event_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'entity_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'event_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'entity_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'entity_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'entity_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'event_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'entity_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'entity_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'entity_mentions', 'dataset_name': 'ECBplus-prep', 'split': 'train'}, {'mention_type': 'entity_mentions', 'dataset_

### Debug problematic mentions

In [10]:
# topics_list = set([mention["topic_id"] for mention in val.mentions])
# subtopics_list = set([mention["subtopic_id"] for mention in val.mentions])

# problematic_mentions = [mention for mention in train.mentions if min(mention['tokens_number_context'])>len(mention["mention_context"])]

# ids= []
# len_context = []
# min_mention_token_number = []
# for mention in problematic_mentions:
#     ids.append(mention["mention_id"])
#     len_context.append(len(mention["mention_context"]))
#     min_mention_token_number.append(min(mention['tokens_number_context']))

# import pandas as pd
# df = pd.DataFrame({"mention_id":ids,"context_length":len_context,"min_mention_token_number":min_mention_token_number})
# # Print the DataFrame in Markdown-friendly format
# markdown_table = df.to_markdown(index=False)
# print(len(df))
# print(markdown_table)

0
| mention_id   | context_length   | min_mention_token_number   |
|--------------|------------------|----------------------------|


## Create dataloaders, device and model

In [9]:
#Subset data in debug for dummy run
train_subset = data.Subset(train, range(8))
val_subset = data.Subset(val, range(8))

In [10]:
# sampler = MultiTaskBatchSampler(        dataset=train_subset,
#         batch_size=32,
#         mix_opt=1,
#         extra_task_ratio=1,
#         current_epoch=1,
#         total_epochs=10,
#         bin_size=64,
#         bin_on=False,
#         bin_grow_ratio=0.5,
#         sampling = "sequential")
        
# train_loader = DataLoader(
#         train_subset,
#         batch_sampler=sampler,
#         pin_memory=True
#         )
val_loader = data.DataLoader(train_subset, batch_size=config.batch_size, shuffle=True, pin_memory=True)
train_loader = data.DataLoader(val_subset, batch_size=config.batch_size, shuffle=False, pin_memory=True)

#device_ids = config.gpu_num
#device = torch.device("cuda:{}".format(device_ids[0]))
device = torch.device("cpu")

logger.info('Init models')
cross_encoder_single = FullCrossEncoder(config).to(device)
#cross_encoder = torch.nn.DataParallel(cross_encoder_single, device_ids=[device])
cross_encoder = cross_encoder_single

loading file vocab.json from cache at C:\Users\sago/.cache\huggingface\hub\models--roberta-large\snapshots\716877d372b884cad6d419d828bac6c85b3b18d9\vocab.json
loading file merges.txt from cache at C:\Users\sago/.cache\huggingface\hub\models--roberta-large\snapshots\716877d372b884cad6d419d828bac6c85b3b18d9\merges.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at None
loading configuration file config.json from cache at C:\Users\sago/.cache\huggingface\hub\models--roberta-large\snapshots\716877d372b884cad6d419d828bac6c85b3b18d9\config.json
Model config RobertaConfig {
  "_name_or_path": "roberta-large",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediat

In [11]:

# Define the names of the layers to freeze *empty to keep roberta frozen
unfreeze_layers = [
]
#Unfreezes Roberta layers in list
for name, param in cross_encoder_single.model.named_parameters():
    param.requires_grad = any(layer_name in name for layer_name in unfreeze_layers)

# Unfreeze the parameters of the linear layers
for param in cross_encoder_single.linear_cdcr.parameters():
    param.requires_grad = True

# Unfreeze the parameters of the linear layers
for param in cross_encoder_single.linear_pi.parameters():
    param.requires_grad = True


In [12]:
## Optimizer and loss function
criterion = get_loss_function(config)
optimizer = get_optimizer(config, [cross_encoder])
scheduler = get_scheduler(optimizer, total_steps=config.epochs * len(train_loader))



In [15]:
len(train_loader)

1

In [14]:
# model_params = list(cross_encoder.parameters())

# # with open('model_parameters.txt', 'w') as f:
# #     for name, param in cross_encoder.named_parameters():
# #         f.write(f"Layer: {name}, Parameters: {param.numel()}\n")

# for name, param in cross_encoder.named_parameters():
#     print(f"Layer: {name}, Parameters: {param.numel()}")

In [17]:
# # Revision prior to training
# cross_encoder.train()
# # Get one batch
# batch_x, batch_y, instance_id, dataset_task = next(iter(train_loader))

# before_sep = [parts[0] for parts in (string.split("[SEP]") for string in batch_x)]
# after_sep = [parts[1] for parts in (string.split("[SEP]") for string in batch_x)]

# bert_tokens = cross_encoder_single.tokenizer(text=before_sep,text_pair=after_sep,add_special_tokens=True,padding=True,truncation=True)
# cross_encoder_single.tokenizer.decode(bert_tokens['input_ids'][0])

In [63]:
##################################################################################
####                    TRAINING
##################################################################################

f1 = []
best_f1 = 0.0
update_interval = 10
train_results_per_epoch= {}
val_results_per_epoch= {}

for epoch in range(config.epochs):
    logger.info('Epoch: {}'.format(epoch))
    accumulate_loss = 0
    number_of_positive_pairs, number_of_pairs = 0, 0
    cross_encoder.train()
    all_train_scores, all_train_labels, all_train_identifiers = [], [], []

    for batch_x, batch_y, instance_id, dataset_task in tqdm(train_loader):
        #Separate sentences from batch to add the tokenizers special tokens
        before_sep = [parts[0] for parts in (string.split("[SEP]") for string in batch_x)]
        after_sep = [parts[1] for parts in (string.split("[SEP]") for string in batch_x)]
        all_train_identifiers.extend([extract_dataset_info_from_instance_id(id) for id in instance_id])

        bert_tokens = cross_encoder_single.tokenizer(text=before_sep,text_pair=after_sep,add_special_tokens=True,padding=True,truncation=True)
        input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
        attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)

        optimizer.zero_grad()
        cross_encoder.zero_grad()
        scores = cross_encoder(input_ids, attention_mask,dataset_task)
        all_train_scores.extend(scores.squeeze(1))

        batch_y = batch_y.view(-1, 1)
        all_train_labels.extend(scores.squeeze(1))

        loss = criterion(scores, batch_y.to(device))
        loss.mean().backward()
        torch.nn.utils.clip_grad_norm_(cross_encoder.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        torch.cuda.empty_cache()


        accumulate_loss += loss.item()

        number_of_positive_pairs += len((batch_y == 1).nonzero())
        number_of_pairs += len(batch_y)

    # list_of_topics = shuffle(list(range(len(training_set.topic_list))))
    # number_of_positive_pairs = 0
    # number_of_pairs = 0
    # for topic_num in tqdm(list_of_topics):
    #
    #
    #
    #
    #     topic = training_set.topic_list[topic_num]
    #     first, second, labels = get_candidate_spans(training_set, topic)
    #
    #     number_of_positive_pairs += len((labels == 1).nonzero())
    #     number_of_pairs += len(labels)
    #
    #
    #     pairwises = CrossEncoderDataset(training_set.mentions_repr, first, second, labels)
    #     generator = data.DataLoader(pairwises, shuffle=True, batch_size=config.batch_size)
    #     loss = fine_tune_bert(cross_encoder, generator, criterion, optimizer, device)
    #     accumulate_loss += loss
    all_train_labels = torch.stack(all_train_labels).tolist()
    all_train_scores = torch.stack(all_train_scores).tolist()
    train_strict_preds = [int(score>0) for score in all_train_scores]
    train_results_per_epoch[epoch] = {'scores':train_strict_preds,'labels':all_train_labels,'identifiers':all_train_identifiers}

    logger.info('Number of positive/total pairs: {}/{}'.format(number_of_positive_pairs, number_of_pairs))
    logger.info('Accumulate loss: {}'.format(accumulate_loss))


    logger.info('Evaluate on the dev set')
    all_scores, all_labels,all_identifiers = [], [], []
    number_of_positive_pairs, number_of_pairs = 0, 0
    cross_encoder.eval()

    for batch_x, batch_y,instance_id,dataset_task in tqdm(val_loader):
        bert_tokens = cross_encoder_single.tokenizer(batch_x, pad_to_max_length=True)
        input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
        attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)
        all_identifiers.extend([extract_dataset_info_from_instance_id(id) for id in instance_id])

        with torch.no_grad():
            scores = cross_encoder(input_ids, attention_mask,dataset_task)

        torch.cuda.empty_cache()
        
        number_of_positive_pairs += len((batch_y == 1).nonzero())
        number_of_pairs += len(batch_y)


    # for topic_num, topic in enumerate(tqdm(dev_set.topic_list)):
    #     dev_topic = dev_set.topic_list[topic_num]
    #     first, second, labels = get_candidate_spans(dev_set, dev_topic)
    #     number_of_positive_pairs += len((labels == 1).nonzero())
    #     number_of_pairs += len(labels)
    #
    #     pairwises = CrossEncoderDataset(dev_set.mentions_repr, first, second, labels)
    #     generator = data.DataLoader(pairwises, shuffle=True, batch_size=config.batch_size)
    #     scores, labels = evaluate_model(cross_encoder, generator, device)
    #     torch.cuda.empty_cache()
    #
        all_scores.extend(scores.squeeze())
        all_labels.extend(batch_y.squeeze())



    all_labels = torch.stack(all_labels)
    all_scores = torch.stack(all_scores)
    strict_preds = (all_scores > 0).to(torch.int)
    
    eval = Evaluation(strict_preds, all_labels.to(device))
    logger.info('Number of predictions: {}/{}'.format(strict_preds.sum(), len(strict_preds)))
    logger.info('Number of positive pairs: {}/{}'.format(len((all_labels == 1).nonzero()),
                                                            len(all_labels)))

    logger.info('Min score: {}'.format(all_scores.min().item()))
    logger.info('Max score: {}'.format(all_scores.max().item()))
    logger.info('Strict - Recall: {}, Precision: {}, F1: {}, Accuracy: {}'.
                format(eval.get_recall(), eval.get_precision(), eval.get_f1(), eval.get_accuracy()))

    val_results_per_epoch[epoch]= {'scores':strict_preds.tolist(),'labels':all_labels.tolist(),'identifiers':all_identifiers}
    
    out_dir = os.path.join(config.model_path, '{}_{}'.format(
        'base' if 'base' in config.roberta_model else 'large', config.batch_size),
        'checkpoint_{}'.format(epoch))
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    current_f1 = eval.get_f1() 
    if current_f1 > best_f1:
        best_f1 = current_f1
        cross_encoder_single.model.save_pretrained(os.path.join(out_dir, 'bert'))
        torch.save(cross_encoder_single.linear_cdcr.state_dict(), os.path.join(out_dir, 'linear_cdcr'))
        torch.save(cross_encoder_single.linear_cdcr.state_dict(), os.path.join(out_dir, 'linear_pi'))

    f1.append(eval.get_f1())
print("F1 values",f1)


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]C:\Users\sago\AppData\Roaming\Python\Python38\site-packages\transformers\tokenization_utils_base.py:2377: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
100%|██████████| 1/1 [00:28<00:00, 28.42s/it]
Configuration saved in models/pairwise_scorers\large_256\checkpoint_0\bert\config.json
Model weights saved in models/pairwise_scorers\large_256\checkpoint_0\bert\pytorch_model.bin
100%|██████████| 1/1 [00:30<00:00, 30.03s/it]

F1 values [1.0, 1.0, 1.0]


In [64]:
import json
# Specify the filename

filename_train = os.path.join(out_dir, "train_results.json")
filename_val = os.path.join(out_dir, "val_results.json")

# Write the JSON data to the file
with open(filename_train, 'w') as json_file:
    json.dump(train_results_per_epoch, json_file, indent=4)

# Write the JSON data to the file
with open(filename_val, 'w') as json_file:
    json.dump(val_results_per_epoch, json_file, indent=4)

{'scores': tensor([1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32),
 'labels': tensor([1., 1., 1., 1., 1., 1., 1., 1.]),
 'identifiers': [{'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'}]}

In [48]:
(torch.tensor(train_results_per_epoch[0]['scores'])> 0).to(torch.int)

tensor([1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)

In [45]:
train_results_per_epoch[0]['scores']

TypeError: '>' not supported between instances of 'list' and 'int'